# PCB Defect Detection — reproducible GPU run
Select Runtime → Change runtime type → GPU. This notebook calls the same fail-fast pipeline used locally.
Real training results are pending. Keep QUICK=True for the first execution; smoke results are not final results.
Download results before the runtime disconnects. See EXPERIMENTS.md for the controlled core runs.


In [ ]:
import os, sys, subprocess
from pathlib import Path

def run(args):
    print(subprocess.list2cmdline([str(a) for a in args]), flush=True)
    subprocess.run([str(a) for a in args], check=True)

PROJECT = Path('/content/PCB-Defect-Detection')
if not PROJECT.exists():
    run(['git', 'clone', 'https://github.com/asadh-74/PCB-Defect-Detection.git', PROJECT])
os.chdir(PROJECT)
assert (PROJECT / 'src').exists(), 'Download the complete updated repository'
run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])
run([sys.executable, '-m', 'pytest', '-q', 'tests'])
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before training'
print(torch.cuda.get_device_name(0))


## Choose the data and run
Upload and extract the dataset (Annotations/ and images/), then set RAW to its root.
Leave RAW=None to try the automatic downloader. Data-provider access can vary.
Use unique RUN_NAME values for variants and keep source split/seed fixed for the core ablations.


In [ ]:
QUICK = True
RAW = None  # Example: '/content/PCB_DATASET' or a Kaggle input path
RUN_NAME = 'smoke' if QUICK else 'grouped_tiles'
SPLIT_MODE = 'board'
REPRESENTATION = 'tiled'  # 'full' for whole-image resizing
AUGMENTATION = 'on'      # 'off' for the augmentation ablation
EPOCHS = 2 if QUICK else 20

args = [sys.executable, 'run_pipeline.py', '--profile', 'quick' if QUICK else 'gpu',
        '--run-name', RUN_NAME, '--split-mode', SPLIT_MODE, '--representation', REPRESENTATION,
        '--augmentation', AUGMENTATION, '--model', 'yolov8n.pt', '--epochs', str(EPOCHS),
        '--batch', '8', '--skip-frcnn']
if RAW is not None:
    args += ['--raw', RAW]
run(args)


## Inspect actual outputs
Quick runs are execution checks. Do not paste their metrics into the report.
Full-board scores compare like evaluation units; tile diagnostics are separate.


In [ ]:
import json
from IPython.display import display, Markdown, Image
result = Path('results') / f'{RUN_NAME}_yolo'
display(Markdown((Path('results') / RUN_NAME / 'comparison.md').read_text()))
display(Image(filename=str(result / 'confusion_matrix.png')))
print(json.loads((result / 'stats.json').read_text()))
for image in sorted((result / 'error_examples').glob('*.jpg'))[:4]:
    display(Image(filename=str(image)))


## Download and preserve the trained model and evidence
The pipeline exports models/yolo_best.pt and its metrics. Unzip the downloaded archive into your local
project to run Streamlit. These are your actual run artifacts; do not substitute COCO starting weights.


In [ ]:
import zipfile
from google.colab import files
archive = Path('/content') / f'pcb_{RUN_NAME}_artifacts.zip'
with archive.open('wb') as target:
    with zipfile.ZipFile(target, 'w', zipfile.ZIP_DEFLATED) as z:
        for directory in ['results', 'weights', 'models']:
            for path in Path(directory).rglob('*'):
                if path.is_file():
                    z.write(path, path.as_posix())
files.download(str(archive))
